# Recommender Systems Track · Matrix Factorization And Embeddings

**Track:** Recommender Systems  
**Objective:** Master the principles and applications of Matrix Factorization And Embeddings.

---



### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Matrix factorization is the RecSys equivalent of embeddings in NLP. User embeddings and item embeddings live in the same latent space — their dot product predicts the rating. This is the EXACT same idea as Word2Vec (NLP/02) and two-tower models (RS/03). Understanding MF gives you a unified framework for recommendations, search, and retrieval.

**Mental Model:** Imagine compressing each user's entire rating history into a short personality profile (a vector of 50 numbers). Similarly, compress each movie into a 50-number DNA. A user will like a movie if their personality aligns with the movie's DNA — measured by the dot product of the two vectors.

**Common Junior Pitfall:** Using too many latent factors (k). With k=500, the model memorizes the training data (overfitting). The Netflix Prize winning solution used k=20-200 with heavy regularization. Start with k=50 and add regularization.

---

## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Intuition: Why Factorize?](#1-intuition-why-factorize)
  * [The Idea](#the-idea)
  * [Why It Works](#why-it-works)
* [2 · SVD: The Classic Approach](#2-svd-the-classic-approach)
* [3 · Learning MF with Gradient Descent](#3-learning-mf-with-gradient-descent)
  * [The Loss Function](#the-loss-function)
* [4 · Visualizing Learned Embeddings](#4-visualizing-learned-embeddings)
* [5 · Connection to Modern Embeddings](#5-connection-to-modern-embeddings)
  * [🎓 DEEP QUESTION ANSWERED](#deep-question-answered)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: Number of latent factors](#q1-number-of-latent-factors)
  * [Q2: Regularization](#q2-regularization)
  * [Q3: MF vs neural network](#q3-mf-vs-neural-network)
* [🔨 Practical Practice](#practical-practice)
  * [Exercise 1: ALS](#exercise-1-als)
  * [Exercise 2: Implicit Feedback](#exercise-2-implicit-feedback)
  * [Exercise 3: Similar Items](#exercise-3-similar-items)


In [ ]:
!pip install -q numpy pandas matplotlib

## 1 · Intuition: Why Factorize?

### The Idea

Decompose the big user-item rating matrix $R$ into two smaller matrices:

$$R \approx U \cdot V^T$$

Where:
- $R$ is $m \times n$ (users × items)
- $U$ is $m \times k$ (user embeddings)
- $V$ is $n \times k$ (item embeddings)
- $k$ is the number of latent factors (typically 20-200)

```
R (200 × 50)  ≈  U (200 × 20)  ×  V^T (20 × 50)
    [sparse]       [user embed]      [item embed]
    10,000 values  4,000 values    1,000 values
```

### Why It Works

The factorization forces the model to find a LOW-RANK approximation — a compressed representation that captures the main patterns. Noise and rare coincidences get discarded. The remaining k dimensions capture the dominant taste patterns.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create movie ratings data (same as RS/01)
np.random.seed(42)
n_users, n_items, n_factors = 200, 50, 20

# Generate data with hidden structure
# True latent factors (unknown to the model)
true_U = np.random.randn(n_users, 5) * 0.5  # 5 hidden dimensions
true_V = np.random.randn(n_items, 5) * 0.5
true_R = 3.0 + true_U @ true_V.T + np.random.randn(n_users, n_items) * 0.3
true_R = np.clip(true_R, 1, 5).round(1)

# Make it sparse (each user rates ~10% of items)
mask = np.random.random((n_users, n_items)) < 0.10
R_observed = true_R * mask

print(f'Matrix Factorization Setup:')
print(f'  Users: {n_users}, Items: {n_items}')
print(f'  Observed ratings: {mask.sum()}')
print(f'  Sparsity: {1 - mask.sum()/(n_users*n_items):.1%}')
print(f'  Latent factors (k): {n_factors}')
print(f'\n  Goal: Learn U ({n_users}x{n_factors}) and V ({n_items}x{n_factors})')
print(f'  such that U[i] dot V[j] predicts R[i,j]')

---
## 2 · SVD: The Classic Approach

In [ ]:
# Truncated SVD — the textbook approach
from numpy.linalg import svd

# Fill missing values with item means (simple imputation)
R_filled = R_observed.copy()
for j in range(n_items):
    col_mean = R_observed[:, j][R_observed[:, j] > 0].mean() if (R_observed[:, j] > 0).any() else 3.0
    R_filled[:, j] = np.where(R_observed[:, j] > 0, R_observed[:, j], col_mean)

# SVD decomposition
U_full, sigma, Vt_full = svd(R_filled, full_matrices=False)

# Truncate to k factors
k = 20
U_k = U_full[:, :k] * np.sqrt(sigma[:k])    # User embeddings
V_k = Vt_full[:k, :].T * np.sqrt(sigma[:k])  # Item embeddings
R_reconstructed = U_k @ V_k.T

# Evaluate: RMSE on observed ratings
predicted = R_reconstructed[mask]
actual = true_R[mask]
rmse = np.sqrt(np.mean((predicted - actual) ** 2))

print(f'SVD Results (k={k} factors):')
print(f'  RMSE on observed ratings: {rmse:.3f}')
print(f'  Singular values: {sigma[:5].round(1)} ... (top 5)')

# Show how many factors are needed
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sigma[:30], 'o-', color='steelblue', markersize=4)
axes[0].set_xlabel('Factor')
axes[0].set_ylabel('Singular Value')
axes[0].set_title('Singular Values (how important each factor is)')
axes[0].grid(alpha=0.3)

cumvar = np.cumsum(sigma**2) / np.sum(sigma**2)
axes[1].plot(cumvar[:30], 'o-', color='coral', markersize=4)
axes[1].axhline(y=0.9, color='green', linestyle='--', label='90% variance')
axes[1].set_xlabel('Number of Factors')
axes[1].set_ylabel('Cumulative Variance Explained')
axes[1].set_title('How many factors do we need?')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f'\nFirst 5 factors capture most variance — the rest is noise.')
print(f'This is why k=20-50 works well in practice.')

---
## 3 · Learning MF with Gradient Descent

SVD requires filling in missing values first (introduces bias). A better approach: only optimize on OBSERVED ratings.

### The Loss Function

$$\min_{U, V} \sum_{(i,j) \in \text{observed}} \left( r_{ij} - \mathbf{u}_i^T \mathbf{v}_j \right)^2 + \lambda(\|\mathbf{u}_i\|^2 + \|\mathbf{v}_j\|^2)$$

In plain English: minimize prediction error on observed ratings + regularization to prevent overfitting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class MatrixFactorization:
    """Matrix Factorization with SGD.
    
    This is the algorithm that won the Netflix Prize.
    User i's predicted rating for item j = u_i dot v_j + b_i + c_j + mu
    
    The biases capture:
    - mu: global average rating
    - b_i: user i's tendency (harsh critic vs easy rater)
    - c_j: item j's tendency (blockbuster vs niche)
    """
    
    def __init__(self, n_users, n_items, k=20, lr=0.005, reg=0.02):
        self.k = k
        self.lr = lr
        self.reg = reg
        
        # Initialize embeddings randomly (small values)
        self.U = np.random.normal(0, 0.1, (n_users, k))  # User embeddings
        self.V = np.random.normal(0, 0.1, (n_items, k))  # Item embeddings
        self.b_u = np.zeros(n_users)  # User biases
        self.b_i = np.zeros(n_items)  # Item biases
        self.mu = 0  # Global mean
    
    def predict(self, user, item):
        """Predict rating: mu + b_u + b_i + u dot v"""
        return self.mu + self.b_u[user] + self.b_i[item] + self.U[user] @ self.V[item]
    
    def fit(self, R_observed, mask, epochs=50):
        """Train with SGD on observed ratings only."""
        self.mu = R_observed[mask].mean()
        
        # Get list of observed (user, item, rating) triples
        users, items = np.where(mask)
        ratings = R_observed[mask]
        
        history = []
        for epoch in range(epochs):
            # Shuffle training data
            indices = np.random.permutation(len(users))
            total_loss = 0
            
            for idx in indices:
                u, i, r = users[idx], items[idx], ratings[idx]
                
                # Prediction error
                pred = self.predict(u, i)
                error = r - pred
                
                # SGD updates
                self.b_u[u] += self.lr * (error - self.reg * self.b_u[u])
                self.b_i[i] += self.lr * (error - self.reg * self.b_i[i])
                
                # Update embeddings
                u_old = self.U[u].copy()
                self.U[u] += self.lr * (error * self.V[i] - self.reg * self.U[u])
                self.V[i] += self.lr * (error * u_old - self.reg * self.V[i])
                
                total_loss += error ** 2
            
            rmse = np.sqrt(total_loss / len(users))
            history.append(rmse)
            if epoch % 10 == 0:
                print(f'  Epoch {epoch:3d}: RMSE = {rmse:.4f}')
        
        return history

# Train
mf = MatrixFactorization(n_users, n_items, k=20, lr=0.01, reg=0.02)
print('Training Matrix Factorization...')
history = mf.fit(R_observed, mask, epochs=50)

# Plot training curve
plt.figure(figsize=(8, 4))
plt.plot(history, lw=2, color='steelblue')
plt.xlabel('Epoch')
plt.ylabel('Training RMSE')
plt.title('Matrix Factorization Training')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Evaluate on held-out data
test_mask = ~mask & (true_R > 0)
if test_mask.sum() > 0:
    test_users, test_items = np.where(test_mask)
    preds = [mf.predict(u, i) for u, i in zip(test_users[:100], test_items[:100])]
    actuals = true_R[test_mask][:100]
    test_rmse = np.sqrt(np.mean((np.array(preds) - actuals) ** 2))
    print(f'\nTest RMSE: {test_rmse:.3f}')

---
## 4 · Visualizing Learned Embeddings

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# Project item embeddings to 2D for visualization
pca = PCA(n_components=2)
items_2d = pca.fit_transform(mf.V)

# Color by genre
genres = ['Action', 'Comedy', 'Drama', 'Sci-Fi', 'Horror', 'Romance']
genre_list = np.random.choice(genres, n_items)  # Same as RS/01
colors = {'Action': 'red', 'Comedy': 'orange', 'Drama': 'blue',
          'Sci-Fi': 'green', 'Horror': 'purple', 'Romance': 'pink'}

plt.figure(figsize=(10, 7))
for genre in genres:
    idx = [i for i, g in enumerate(genre_list) if g == genre]
    plt.scatter(items_2d[idx, 0], items_2d[idx, 1],
                c=colors[genre], label=genre, s=60, alpha=0.7)

plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.title('Item Embeddings (colored by genre)')
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print('Items cluster by genre in the latent space!')
print('The model DISCOVERED genre similarity from rating patterns alone.')
print('Nobody told it about genres — it found this structure itself.')
print('\nThis is the EXACT same phenomenon as word embeddings (NLP/02):')
print('  Word2Vec: similar words have similar vectors')
print('  MF:       similar movies have similar vectors')

---
## 5 · Connection to Modern Embeddings

Matrix factorization is the same idea as:

| Domain | Method | User Embedding | Item Embedding |
|--------|--------|---------------|----------------|
| RecSys | MF (this notebook) | User latent vector | Item latent vector |
| NLP | Word2Vec (NLP/02) | Context word vector | Target word vector |
| Search | Two-Tower (RS/03) | Query embedding | Document embedding |
| RAG | Retrieval (NB28) | Query embedding | Chunk embedding |

All of them: **learn two sets of vectors, predict relevance by dot product.**

### 🎓 DEEP QUESTION ANSWERED

**Q:** *Why does breaking a big matrix into two small ones discover that Matrix fans like Inception?*

**A:** The factorization forces a BOTTLENECK — k dimensions must explain millions of ratings. The only way to do this efficiently is to discover the underlying PATTERNS. If many users who liked Matrix also liked Inception, the model must capture this correlation in its k dimensions. The user vectors for Matrix fans will be similar, and the item vectors for Matrix and Inception will be similar — producing high predicted ratings. The bottleneck forces generalization, which IS pattern discovery.

---

## ✅ Knowledge Check

### Q1: Number of latent factors
What happens if k (latent factors) is too large? What if it's too small?

<details><summary>👉 Answer</summary>

Too large (k=500): model memorizes training ratings (overfitting), poor generalization. The embeddings capture noise, not signal. Too small (k=2): model can't capture enough taste dimensions — it might conflate "likes action" with "likes old movies" because there aren't enough dimensions to separate them. Sweet spot: k=20-200 with regularization.
</details>

### Q2: Regularization
Why add λ(||u||² + ||v||²) to the loss? What happens without it?

<details><summary>👉 Answer</summary>

Without regularization, embedding vectors grow arbitrarily large to fit training data perfectly. This overfits to noise in the ratings (a user giving 5 stars might have been in a good mood). Regularization keeps vectors small, forcing the model to use the DIRECTION (similarity) rather than magnitude. This is L2 regularization — the same concept as in logistic regression (ML/02).
</details>

### Q3: MF vs neural network
MF uses a DOT PRODUCT to combine user and item embeddings. What advantage would a NEURAL NETWORK have?

<details><summary>👉 Answer</summary>

Dot product is a LINEAR interaction — it can only capture linear relationships between latent factors. A neural network can learn NON-LINEAR interactions (e.g., "users who like BOTH action AND romance like this movie, but not users who like only one"). This is Neural Collaborative Filtering (RS/03). The tradeoff: neural models are harder to train and need more data.
</details>

---

## 🔨 Practical Practice

### Exercise 1: ALS
Implement Alternating Least Squares: fix U and solve for V (closed-form), then fix V and solve for U, alternate. Compare convergence speed to SGD.

### Exercise 2: Implicit Feedback
Modify MF for implicit feedback (clicks, views — no ratings). Use the Weighted MF approach: observed interactions have weight 1, unobserved have weight 0, and optimize confidence-weighted loss.

### Exercise 3: Similar Items
Using the learned V matrix, find the 5 most similar items to each item (cosine similarity of embeddings). Build a "Because you watched X" feature.

---

**Next →** RS 03: Deep Learning for Recommendations